# Week 10 Lab · Vector Autoregression (VAR): Explain the Evidence
**수업 실습 v7 · 제출 없음 · 일반 런타임 사용 · 약 55분**

Vector Autoregression (VAR, 벡터자기회귀)는 여러 계열의 과거를 함께 사용하는 예측 모형입니다. Autoregressive (AR, 자기회귀) 모형은 한 계열의 자기 과거를 사용합니다.

이번에는 공식을 세 줄만 연습합니다. 코드 전체를 이해하기보다 **출력에서 무엇을 말할 수 있는지**에 집중하세요.

| 순서 | 읽을 출력 / 할 일 | 강의 v6 |
|---|---|---|
| 1 | 두 과거로 다음 값 계산 · Code 4 연습 | 4–6장 |
| 2 | CORE_READ: 안정성·잔차·Granger p값 | 7–8장 |
| 3 | 기준 경로 대비 반응 · Code 5 연습 | 9장 |
| 4 | ORDER_READ: 두 순서의 즉시 반응 비교 | 10장 |
| 5 | VALIDATION으로 선택 → FINAL에서 비교 · Code 6 연습 | 11–12장 |
| 6 | 근거·숫자·평가구간·한계로 결론 쓰기 | 13–15장 |

**File → Save a copy in Drive** 후 위에서부터 실행하세요. `None` 세 곳만 채웁니다. 미완성은 안내만 표시하며 뒤 분석은 계속됩니다. 문법 오류는 수정해야 합니다. 아래 접힌 정답은 연습용이며, 과제 파일이 아닙니다.

모의자료는 원본 v5와 동일한 **400개 관측값**입니다. 이번에는 앞 280개를 training, 다음 60개를 validation, 마지막 60개를 final로 나눕니다. 과제 A3의 240/60/60 자료와는 별개입니다. 모형은 앞 구간만 보고 진단합니다. final을 공개한 뒤에는 선택을 바꾸지 마세요. 이는 학습 규칙이며 자료가 암호화되어 숨겨져 있는 것은 아닙니다.

F·자유도 계산, 고유값 손계산, Cholesky 분해 구현은 필수 실습에서 제외합니다.

In [ ]:
import sys, importlib.util, subprocess
required = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib', 'scipy':'scipy', 'statsmodels':'statsmodels==0.14.5'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels
from statsmodels.tsa.api import VAR
from IPython.display import display
np.set_printoptions(precision=4, suppress=True)
GREEN, GOLD = '#1E4A32', '#D9AE3A'
print('numpy', np.__version__, '| pandas', pd.__version__, '| statsmodels', statsmodels.__version__)
print('No GPU or data download required. Missing packages need internet.')

## 1. 두 과거로 다음 값 계산하기 / One-step prediction
행은 예측할 변수, 열은 누구의 과거인지 나타냅니다. `@`는 행렬곱입니다.

$$\widehat{\mathbf{x}}_t=\mathbf{a}+\Phi\mathbf{x}_{t-1}$$

강의의 작은 예제는 상수가 0입니다. 일반 적합 모형에서는 상수를 더합니다. 예측에는 아직 모르는 미래 충격을 넣지 않습니다.

**적합과 오차:** Residual Sum of Squares (RSS, 잔차제곱합)는 실제값−적합값을 제곱해 더한 값입니다. Ordinary Least Squares (OLS, 보통최소제곱법)는 각 방정식의 RSS가 가장 작아지도록 계수를 추정합니다.

In [ ]:
def one_step(intercept, matrix, last):
    prediction = None  # TODO 1 / A3 Code 4: intercept plus matrix times last
    return prediction

toy = np.array([[.5,.2],[.1,.4]])
checks = {}
def check(name, fn, cases):
    try:
        for args, expected in cases:
            got = fn(*args)
            if got is None:
                checks[name] = False
                print(name, ': blank. Fill None; analysis can continue.')
                return
            np.testing.assert_allclose(got, expected, atol=1e-10)
        checks[name] = True
        print(name, ': passed')
    except Exception as exc:
        checks[name] = False
        print(name, ': check the formula.', type(exc).__name__, str(exc)[:140])

check('one_step', one_step, [((np.zeros(2),toy,np.array([2.,1.])),np.array([1.2,.6])),
    ((np.array([1.,-1.]),toy,np.array([2.,1.])),np.array([2.2,-.4]))])

### 제공 코드: 같은 모의자료 생성 및 구간 분리
원본 v5의 시드·계수·충격·400개 자료를 유지했습니다. 아래 코드는 수정하지 않아도 됩니다. 전체 final 구간은 아직 그리지 않습니다.

In [ ]:
PHI_TRUE = np.array([[.5,0.],[.6,.3]])
SIGMA_TRUE = np.array([[1.,.5],[.5,1.]])
N, BURN = 400, 50
rng = np.random.default_rng(8)
eps = rng.standard_normal((N+BURN,2)) @ np.linalg.cholesky(SIGMA_TRUE).T
X = np.zeros((N+BURN,2))
for t in range(1,N+BURN):
    X[t] = PHI_TRUE @ X[t-1] + eps[t]
df = pd.DataFrame(X[BURN:],columns=['x','y'])
TRAIN_END, VALID_END = 280,340
train = df.iloc[:TRAIN_END]
fits = {p:VAR(train).fit(p) for p in [1,2]}
ax=train.plot(color=[GREEN,GOLD],figsize=(10,3),title='Training observations only (0–279)')
ax.set_xlabel('observation'); ax.set_ylabel('simulated value'); plt.show()
print('Training: 0–279 | Validation: 280–339 | Final: 340–399')

## 2. 후보로 남길 근거가 있는가? / Read CORE_READ
VAR1의 **stable**, **residual p-value**, **x → y Granger p-value**를 읽으세요. 과제 Q2와 같은 종류의 판단입니다.

- stable: 한 번의 충격이 장기적으로 사라지는가?
- 잔차 검정: 남은 시차 의존성이 탐지되는가? 큰 p값도 모형이 옳다는 증명은 아닙니다.
- Granger: 다른 포함 시차를 조건으로, y 식에서 x의 모든 과거 계수가 0인가? 기각해도 개입 인과효과를 증명하지 않습니다.

**작성:** 후보를 유지/재검토할 근거와 한계를 두 문장으로 적으세요. 작은 Granger p값만으로 최종 모델을 고르지 않습니다.

아래 잔차 그림의 autocorrelation function (ACF, 자기상관함수)은 시차별 상관을 보여줍니다.

In [ ]:
rows=[]
for p,fit in fits.items():
    rows.append({'model':f'VAR{p}', 'stable':fit.is_stable(),
        'residual p-value':fit.test_whiteness(nlags=10,adjusted=True).pvalue,
        'Granger x → y p-value':fit.test_causality('y',['x'],kind='f').pvalue})
CORE_READ=pd.DataFrame(rows).set_index('model')
display(CORE_READ)
fig,axs=plt.subplots(1,2,figsize=(10,3))
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(fits[1].resid['x'],lags=12,ax=axs[0],title='VAR1 residual x')
plot_acf(fits[1].resid['y'],lags=12,ax=axs[1],title='VAR1 residual y')
plt.tight_layout();plt.show()

## 3. 충격 후 차이는 어떻게 전달되는가? / Response difference
Impulse Response Function (IRF, 충격반응함수)은 충격을 준 경로와 기준 경로의 차이를 추적합니다.

강의의 toy VAR(1)에 `(1, 0)`을 한 번 더한 경로와 더하지 않은 경로를 비교합니다.

$$\mathbf{d}_{h}=\Phi\mathbf{d}_{h-1}$$

상수는 두 경로를 빼면 없어지므로 **더하지 않습니다**. 이 한 줄은 VAR(1)용입니다. VAR(2)에는 두 시차를 이용한 재귀가 필요하며 직접 구현할 필요 없습니다.

In [ ]:
def next_response(matrix, previous):
    response = None  # TODO 2 / A3 Code 5: matrix times previous response
    return response

check('next_response',next_response,[((toy,np.array([1.,0.])),np.array([.5,.1])),
    ((toy,np.array([.5,.1])),np.array([.27,.09]))])
# Provided plot: it remains available when the exercise is incomplete.
path=np.stack([np.linalg.matrix_power(toy,h) @ np.array([1.,0.]) for h in range(11)])
display(pd.DataFrame(path[:4],columns=['x response','y response']).rename_axis('h'))
plt.figure(figsize=(9,3));plt.plot(path[:,0],'-o',color=GREEN,label='x');plt.plot(path[:,1],'-o',color=GOLD,label='y')
plt.xlabel('h');plt.ylabel('difference from baseline');plt.legend();plt.show()

## 4. 어떤 즉시 반응을 0으로 정했는가? / Read ORDER_READ
**y의 표준화 충격에 대한 x의 h=0 반응**만 비교하세요. 같은 training 자료와 VAR1을 사용합니다.

`x first`는 y 충격에 대한 x의 즉시 반응을 0으로 둡니다. `y first`는 그 반응을 허용합니다. 순서마다 충격의 정의와 크기도 달라집니다. 같은 reduced-form 충격을 고정한 비교가 아닙니다.

**작성:** 두 값을 보고하고 어느 0이 가정인지 설명하세요. 실제 시간 순서를 관찰했거나 인과효과가 없음을 검증한 것이 아닙니다.

In [ ]:
r_xy=fits[1]
r_yx=VAR(train[['y','x']]).fit(1)
ir_xy=r_xy.irf(1).orth_irfs
ir_yx=r_yx.irf(1).orth_irfs
ORDER_READ=pd.DataFrame([
    ['x first',ir_xy[0,0,1],ir_xy[0,1,1]],
    ['y first',ir_yx[0,1,0],ir_yx[0,0,0]]],
    columns=['ordering','x response at h=0','y impact scale']).set_index('ordering')
display(ORDER_READ)
print('Each shock is +1 standardized unit under its own ordering.')

## 5. 검증 구간에서 선택하기 / Choose before the final evaluation
VAR1, VAR2, AR1을 **같은 y와 같은 예측시점**에서 비교합니다. 매번 다음 한 관측값만 예측하며 그 전에 알려진 자료로 다시 적합합니다. 선택 기준은 검증 구간의 Root Mean Squared Error (RMSE, 평균제곱근오차)입니다.

$$\operatorname{RMSE}=\sqrt{\operatorname{mean}((\mathrm{actual}-\mathrm{forecast})^2)}$$

작을수록 좋고 y와 단위가 같습니다. final을 공개한 뒤에는 시차를 다시 선택하지 마세요.

In [ ]:
def rmse(actual,predicted):
    error=actual-predicted
    result = None  # TODO 3 / A3 Code 6: square, mean, square root
    return result

check('rmse',rmse,[((np.array([1.,4.,-1.]),np.array([0.,2.,1.])),np.sqrt(3.)),
    ((np.array([0.,2.]),np.array([0.,2.])),0.)])

In [ ]:
# Provided evaluation helper. Fit on data strictly before each forecast origin.
def forecasts(data,start,stop,models):
    predictions={name:[] for name in models}
    for origin in range(start,stop):
        past=data.iloc[:origin]
        for name in models:
            if name=='AR1':
                y=past['y'].to_numpy()
                z=np.column_stack([np.ones(len(y)-1),y[:-1]])
                a,b=np.linalg.lstsq(z,y[1:],rcond=None)[0]
                value=a+b*y[-1]
            else:
                lag=int(name[-1]);fitted=VAR(past).fit(lag)
                value=fitted.forecast(past.to_numpy()[-lag:],1)[0,1]
            predictions[name].append(value)
    return {name:np.asarray(v) for name,v in predictions.items()}

def error_table(actual,predictions):
    return pd.DataFrame([{'model':name,'RMSE':np.sqrt(np.mean((actual-values)**2))}
                         for name,values in predictions.items()]).set_index('model')

validation_predictions=forecasts(df,TRAIN_END,VALID_END,['VAR1','VAR2','AR1'])
VALIDATION=error_table(df['y'].iloc[TRAIN_END:VALID_END].to_numpy(),validation_predictions)
display(VALIDATION)
print('60 one-step forecasts: observations 280–339. Choose one VAR before revealing final.')

### 선택 기록
아래 두 문자열을 채우고 이유를 먼저 적으세요. 그 뒤 `reveal_final=True`로 바꾸세요. final은 기본적으로 출력되지 않습니다. 공개 후 같은 런타임에서는 선택 변경을 막습니다. 런타임 재시작으로 잠금이 사라질 수 있으므로 엄격한 blind test가 아닙니다.

In [ ]:
preferred = ''       # 'VAR1' or 'VAR2'
reason = ''          # cite validation RMSE compared with AR1
reveal_final = False

if not reveal_final:
    print('Final results not displayed. Record your choice and reason first.')
elif preferred not in ['VAR1','VAR2'] or not reason.strip():
    print('Enter VAR1 or VAR2 and a reason before revealing final.')
elif '_locked_choice' in globals() and (preferred,reason)!=_locked_choice:
    print('Choice already recorded. Restore the original choice and reason:',_locked_choice)
else:
    if '_locked_choice' not in globals():
        _locked_choice=(preferred,reason)
    final_predictions=forecasts(df,VALID_END,len(df),[preferred,'AR1'])
    FINAL=error_table(df['y'].iloc[VALID_END:].to_numpy(),final_predictions)
    display(FINAL)
    print('Choice:',_locked_choice,'| Final: 60 one-step forecasts, observations 340–399')
    print('Earlier final observations may enter later fits; current/future outcomes never do.')

## 6. 결론 쓰기 / Explain the evidence
1. 진단 결과로 이 모델을 후보로 남길 이유와 한계는 무엇인가?
2. validation에서 선택한 VAR의 RMSE와 AR1의 RMSE는 각각 얼마인가?
3. final에서 같은 비교가 어떻게 달라졌는가? 숫자와 평가구간을 쓰고 선택은 바꾸지 않는다.
4. Granger p값만으로 알 수 없던 무엇을 최종 비교가 보여주는가?

**텀프로젝트 연결:** 실제 소비·소득, 동일 예측시점과 **4분기 horizon**, rolling origins, 잔차 진단, 예측구간의 coverage와 폭을 사용합니다. 오늘 실습은 simulated **one-step** 예측으로 절차를 연습합니다. 텀프에는 제공된 프로젝트 노트북을 사용하세요.

여기까지가 필수입니다. 아래는 정답 확인과 기존 실증 결과 재현용 선택 참고입니다.

## 선택 참고 · 연습 정답
연습용 정답은 접혀 있어도 열어볼 수 있습니다. 먼저 위 빈칸과 자가검증을 실행하세요. 아래 셀은 위 함수를 덮어쓰지 않으며, `show_solutions=True`일 때만 세 줄을 표시합니다.

In [ ]:
#@title Practice solutions (open only after trying) { display-mode: "form" }
show_solutions = False
if show_solutions:
    print('TODO 1: intercept + matrix @ last')
    print('TODO 2: matrix @ previous')
    print('TODO 3: np.sqrt(np.mean(error**2))')

## 선택 참고 · 실제 데이터의 기존 결과 재현
원본 v5에 들어 있던 US consumption/income 198분기를 그대로 사용합니다. `run_real_example=True`로 바꾸면 기존 강의의 계수·순서·마지막20분기 RMSE를 재현합니다. 자료 다운로드는 없습니다. 모의자료의 final 평가와 별개의 예제입니다.

Source: Forecasting: Principles and Practice, the Pythonic Way (FPP-py), §12.3, [Vector autoregressions](https://otexts.com/fpppy/). 사용 데이터는 제공된 원본 노트북의 내장 자료입니다.

In [ ]:
#@title Optional: reproduce the existing real-data results { display-mode: "form" }
run_real_example = False
US_CHANGE = """quarter,consumption,income
1970Q1,0.6186,1.0448
1970Q2,0.452,1.2256
1970Q3,0.8729,1.5852
1970Q4,-0.2718,-0.2395
1971Q1,1.9013,1.9759
1971Q2,0.9149,1.4459
1971Q3,0.7941,0.5211
1971Q4,1.6456,1.1592
1972Q1,1.3112,0.4569
1972Q2,1.8858,1.0334
1972Q3,1.5293,1.9263
1972Q4,2.3196,3.9299
1973Q1,1.8093,0.7556
1973Q2,-0.0451,0.8404
1973Q3,0.3514,0.451
1973Q4,-0.2987,1.0882
1974Q1,-0.8987,-1.6742
1974Q2,0.3594,-0.9344
1974Q3,0.4286,0.0903
1974Q4,-1.4725,-0.1337
1975Q1,0.8352,-0.1505
1975Q2,1.6315,4.5219
1975Q3,1.4112,-1.4643
1975Q4,1.0696,0.7787
1976Q1,1.9797,1.2003
1976Q2,0.9158,0.5664
1976Q3,1.0474,0.7772
1976Q4,1.2891,0.6325
1977Q1,1.1718,0.2245
1977Q2,0.5408,0.9355
1977Q3,0.9411,1.38
1977Q4,1.4923,1.8934
1978Q1,0.5602,0.781
1978Q2,2.1153,1.0529
1978Q3,0.419,0.7748
1978Q4,0.8037,0.6557
1979Q1,0.5166,1.2028
1979Q2,-0.065,-0.9082
1979Q3,0.9684,0.4532
1979Q4,0.2485,0.176
1980Q1,-0.1431,0.3455
1980Q2,-2.2778,-0.883
1980Q3,1.091,1.117
1980Q4,1.342,1.3674
1981Q1,0.3344,-0.14
1981Q2,0.0073,0.1166
1981Q3,0.4615,2.1335
1981Q4,-0.6864,0.1098
1982Q1,0.7339,0.2411
1982Q2,0.3041,0.6015
1982Q3,0.6681,0.4787
1982Q4,1.7365,0.3136
1983Q1,0.9838,0.9392
1983Q2,2.0158,0.6907
1983Q3,1.7773,1.4894
1983Q4,1.581,2.2028
1984Q1,0.8265,1.9778
1984Q2,1.3913,1.6473
1984Q3,0.7389,1.3115
1984Q4,1.2871,0.813
1985Q1,1.6848,-0.2613
1985Q2,0.9138,1.9289
1985Q3,1.8815,-0.3138
1985Q4,0.2161,1.0524
1986Q1,0.87,1.3064
1986Q2,1.0838,1.2708
1986Q3,1.7536,0.6355
1986Q4,0.6115,0.1522
1987Q1,0.0998,0.9171
1987Q2,1.3555,-0.9403
1987Q3,1.1348,1.7741
1987Q4,0.2113,1.3395
1988Q1,1.7511,1.4562
1988Q2,0.7355,1.0243
1988Q3,0.8846,0.9619
1988Q4,1.1306,0.939
1989Q1,0.4629,1.1321
1989Q2,0.4657,-0.2952
1989Q3,0.9634,0.6621
1989Q4,0.4337,0.8429
1990Q1,0.8381,0.765
1990Q2,0.3042,0.6935
1990Q3,0.3954,-0.0354
1990Q4,-0.7637,-0.8564
1991Q1,-0.3767,0.2496
1991Q2,0.8253,0.7289
1991Q3,0.4974,0.4074
1991Q4,-0.0501,0.8279
1992Q1,1.855,1.9925
1992Q2,0.6765,0.9868
1992Q3,1.0662,0.4628
1992Q4,1.1848,0.4544
1993Q1,0.3827,0.3734
1993Q2,0.9055,0.3089
1993Q3,1.1068,0.0871
1993Q4,0.874,0.6926
1994Q1,1.1373,0.6582
1994Q2,0.7751,1.0021
1994Q3,0.7596,0.5815
1994Q4,1.0753,1.4247
1995Q1,0.2486,0.8512
1995Q2,0.8728,0.2831
1995Q3,0.9022,0.9121
1995Q4,0.6951,0.5827
1996Q1,0.9181,0.9424
1996Q2,1.07,0.9127
1996Q3,0.6036,0.8065
1996Q4,0.7934,0.5312
1997Q1,1.0486,0.9479
1997Q2,0.4492,0.8823
1997Q3,1.6894,1.1698
1997Q4,1.1881,1.462
1998Q1,1.0065,2.0834
1998Q2,1.7456,1.3656
1998Q3,1.2956,1.0024
1998Q4,1.4344,0.7885
1999Q1,0.9656,0.8999
1999Q2,1.4475,0.225
1999Q3,1.1224,0.6799
1999Q4,1.4546,1.4908
2000Q1,1.5246,1.8964
2000Q2,0.9612,1.0998
2000Q3,0.9772,1.1545
2000Q4,0.8564,0.3568
2001Q1,0.3995,0.9021
2001Q2,0.2361,-0.1748
2001Q3,0.3421,2.2812
2001Q4,1.4842,-1.2718
2002Q1,0.3056,2.2292
2002Q2,0.5121,0.6747
2002Q3,0.7219,-0.0872
2002Q4,0.5731,0.592
2003Q1,0.4692,0.2349
2003Q2,1.1168,1.2115
2003Q3,1.4562,1.6756
2003Q4,0.6758,0.2631
2004Q1,0.8858,0.4595
2004Q2,0.6202,1.1394
2004Q3,1.0998,0.6508
2004Q4,1.0818,1.2551
2005Q1,0.5818,-1.1794
2005Q2,1.1282,0.9455
2005Q3,0.966,0.2952
2005Q4,0.2965,1.2728
2006Q1,1.1089,1.9267
2006Q2,0.5053,0.2442
2006Q3,0.6053,0.2584
2006Q4,0.9631,1.311
2007Q1,0.558,0.8484
2007Q2,0.1852,0.2557
2007Q3,0.562,0.0888
2007Q4,0.2844,0.068
2008Q1,-0.2661,0.2774
2008Q2,0.162,1.8135
2008Q3,-0.7516,-2.124
2008Q4,-0.935,0.8673
2009Q1,-0.2261,-0.43
2009Q2,-0.4058,1.0693
2009Q3,0.6875,-1.1372
2009Q4,-0.1484,-0.0246
2010Q1,0.4932,0.5606
2010Q2,0.7922,1.6342
2010Q3,0.6969,0.7094
2010Q4,0.6592,0.5591
2011Q1,0.4244,1.0139
2011Q2,0.2212,-0.2179
2011Q3,0.3908,0.4356
2011Q4,0.1826,0.3105
2012Q1,0.7998,1.8546
2012Q2,0.1483,0.905
2012Q3,0.163,-0.7184
2012Q4,0.4472,2.7326
2013Q1,0.5172,-4.0844
2013Q2,0.0719,0.7422
2013Q3,0.4059,0.4314
2013Q4,0.858,0.3943
2014Q1,0.3932,1.3833
2014Q2,1.089,1.3529
2014Q3,1.0703,1.1606
2014Q4,1.2054,1.313
2015Q1,0.8273,1.1293
2015Q2,0.7928,0.7496
2015Q3,0.7594,0.7324
2015Q4,0.4468,0.3167
2016Q1,0.7783,0.6661
2016Q2,0.7156,-0.1066
2016Q3,0.6349,0.4473
2016Q4,0.6185,0.598
2017Q1,0.5933,1.192
2017Q2,0.5932,0.6648
2017Q3,0.5847,0.5797
2017Q4,1.1331,0.9072
2018Q1,0.417,1.6657
2018Q2,0.9831,0.6618
2018Q3,0.8532,0.8063
2018Q4,0.3565,0.6951
2019Q1,0.2829,1.1008
2019Q2,1.1135,0.5934
"""
if run_real_example:
    from io import StringIO
    us=pd.read_csv(StringIO(US_CHANGE),index_col='quarter')
    us.index=pd.PeriodIndex(us.index,freq='Q')
    real_fit=VAR(us).fit(1)
    print('Coefficient estimates');display(real_fit.params)
    first_income=VAR(us[['income','consumption']]).fit(1).irf(1).orth_irfs[:,1,0]
    first_consumption=real_fit.irf(1).orth_irfs[:,0,1]
    display(pd.DataFrame([first_income,first_consumption],index=['income first','consumption first'],columns=['h=0','h=1']))
    errors={name:[] for name in ['VAR1','AR1','mean']}
    for t in range(len(us)-20,len(us)):
        past=us.iloc[:t].to_numpy(); actual=us.iloc[t].to_numpy()
        var=VAR(past).fit(1).forecast(past[-1:],1)[0]
        ar=[]
        for j in range(2):
            z=np.column_stack([np.ones(t-1),past[:-1,j]])
            a,b=np.linalg.lstsq(z,past[1:,j],rcond=None)[0]
            ar.append(a+b*past[-1,j])
        for name,pred in [('VAR1',var),('AR1',np.array(ar)),('mean',past.mean(axis=0))]:
            errors[name].append(actual-pred)
    REAL_RMSE=pd.DataFrame({name:np.sqrt(np.mean(np.asarray(e)**2,axis=0)) for name,e in errors.items()},index=['consumption','income']).T
    display(REAL_RMSE)
    print('Same original 198 observations; last 20 quarters, rolling one-step evaluation.')
else:
    print('Optional real-data reproduction skipped.')

In [ ]:
import platform
print('Python:',sys.version.split()[0],platform.platform())
print('numpy:',np.__version__,'pandas:',pd.__version__,'statsmodels:',statsmodels.__version__)
print('Seed 8 | train 280 / validation 60 / final 60 | exercise checks:',checks)
print('For help, post the cell number and final error message on the course board.')